In [ ]:
import pickle
import numpy as np
import sys
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance
sys.path.append('../../')
# out_path = 'qm9_noH_1000_kabsch_traj_interpolator_finetune_final_539_25_evalout.pkl'
# with open(out_path, 'rb') as f:
#     data = pickle.load(f)
#     print(data.keys())  # smiles
# smiles_1 = list(data.keys())
# print(len(data.keys()))  # 1000

# out_path = '20250511_1851_multi_egnn_51_evalout.pkl '
out_path = 'mdbaseline_qm9_evalout.pkl'  # place the _evalout.pkl in the working dir
with open(out_path, 'rb') as f:
    data = pickle.load(f)
    print(data.keys())  # smiles
smiles = list(data.keys())
print(len(data.keys()))  # 1000

In [ ]:
rep_nums = [960, 768, 576, 384, 192, 100, 50]
rep_names = ['960points', '768points', '576points', '384points', '192points', '100points', '50points']
rep_nums = [500, 400, 300, 200, 100, 80, 50]
rep_names = ['500points', '400points', '300points', '200points', '100points', '80points', '50points']

In [ ]:
data['NC(=O)NCC[C@H]1CO1'].keys()

In [ ]:
# data['CCCn1c(N)c(N(CCOC)C(=O)c2ccccc2)c(=O)[nH]c1=O'][f'JSD_torsion_md_{rep_names[i]}']

In [ ]:
# data['CCCn1c(N)c(N(CCOC)C(=O)c2ccccc2)c(=O)[nH]c1=O']['JSD_TICA_md_100points']

In [ ]:
jsd_torsion = [[] for _ in range(len(rep_nums))]
jsd_bond_length = [[] for _ in range(len(rep_nums))]
jsd_bond_angle = [[] for _ in range(len(rep_nums))]
jsd_tica_0 = [[] for _ in range(len(rep_nums))]
jsd_msm = [[] for _ in range(len(rep_nums))]

In [ ]:
for smile in smiles:
    for i, rep_num in enumerate(rep_nums):
        jsd_torsion[i] += list(data[smile][f'JSD_torsion_md_{rep_names[i]}'].values())
        jsd_bond_length[i]+=data[smile][f'JSD_bond_length_md_{rep_names[i]}']
        jsd_bond_angle[i]+=data[smile][f'JSD_bond_angle_md_{rep_names[i]}']
        jsd_tica_0[i].append(data[smile][f'JSD_TICA_md_{rep_names[i]}']['TICA-0'])
        if 'JSD_msm_md_' + rep_names[i] in data[smile].keys():
            jsd_msm[i].append(data[smile][f'JSD_msm_md_{rep_names[i]}'])


In [ ]:
np.array(jsd_bond_length[1]).shape

In [ ]:
import csv

header = [''] + rep_names
rows = [
    ['jsd_bond_angle_mean'] + [np.mean(jsd_bond_angle[j]) if len(jsd_bond_angle[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_bond_angle_std'] + [np.std(jsd_bond_angle[j]) if len(jsd_bond_angle[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_bond_angle_max'] + [np.max(jsd_bond_angle[j]) if len(jsd_bond_angle[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_bond_angle_min'] + [np.min(jsd_bond_angle[j]) if len(jsd_bond_angle[j]) > 0 else float('nan') for j in range(len(rep_names))],

    ['jsd_bond_length_mean'] + [np.mean(jsd_bond_length[j]) if len(jsd_bond_length[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_bond_length_std'] + [np.std(jsd_bond_length[j]) if len(jsd_bond_length[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_bond_length_max'] + [np.max(jsd_bond_length[j]) if len(jsd_bond_length[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_bond_length_min'] + [np.min(jsd_bond_length[j]) if len(jsd_bond_length[j]) > 0 else float('nan') for j in range(len(rep_names))],

    ['jsd_torsion_mean'] + [np.mean(jsd_torsion[j]) if len(jsd_torsion[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_torsion_std'] + [np.std(jsd_torsion[j]) if len(jsd_torsion[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_torsion_max'] + [np.max(jsd_torsion[j]) if len(jsd_torsion[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_torsion_min'] + [np.min(jsd_torsion[j]) if len(jsd_torsion[j]) > 0 else float('nan') for j in range(len(rep_names))],

    ['jsd_tica_0_mean'] + [np.mean(jsd_tica_0[j]) if len(jsd_tica_0[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_tica_0_std'] + [np.std(jsd_tica_0[j]) if len(jsd_tica_0[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_tica_0_max'] + [np.max(jsd_tica_0[j]) if len(jsd_tica_0[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_tica_0_min'] + [np.min(jsd_tica_0[j]) if len(jsd_tica_0[j]) > 0 else float('nan') for j in range(len(rep_names))],

    ['jsd_msm_mean'] + [np.mean(jsd_msm[j]) if len(jsd_msm[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_msm_std'] + [np.std(jsd_msm[j]) if len(jsd_msm[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_msm_max'] + [np.max(jsd_msm[j]) if len(jsd_msm[j]) > 0 else float('nan') for j in range(len(rep_names))],
    ['jsd_msm_min'] + [np.min(jsd_msm[j]) if len(jsd_msm[j]) > 0 else float('nan') for j in range(len(rep_names))]
]

csv_file = f'{out_path}_rep_statistics.csv'
with open(csv_file, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(header)
    writer.writerows(rows)

print(f"Rep statistics saved to {csv_file}")

In [ ]:
bond_angle_jsd = []
bond_angle_jsd_md = []
bond_angle_jsd_top1 = []
bond_angle_jsd_top3 = []
bond_angle_jsd_top5 = []
bond_angle_jsd_top10 = []
bond_angle_w1 = []
bond_angle_w1_md = []
bond_length_jsd = []
bond_length_jsd_top1 = []
bond_length_jsd_top3 = []
bond_length_jsd_top5 = []
bond_length_jsd_top10 = []
bond_length_jsd_md = []
bond_length_w1 = []
bond_length_w1_md = []
torsion_jsd = []
torsion_jsd_top1 = []
torsion_jsd_top3 = []
torsion_jsd_top5 = []
torsion_jsd_top10 = []
torsion_jsd_md = []
torsion_w1 = []
torsion_w1_md = []
tica_0_jsd = []
tica_0_jsd_top1 = []
tica_0_jsd_top3 = []
tica_0_jsd_top5 = []
tica_0_jsd_top10 = []
tica_0_jsd_md = []
tica_0_w1 = []
tica_0_w1_md = []
tica_01_jsd = []
tica_01_jsd_top1 = []
tica_01_jsd_top3 = []
tica_01_jsd_top5 = []
tica_01_jsd_top10 = []
tica_01_jsd_md = []
decorrelation_jsd = []
decorrelation_jsd_md = []
decorrelation_w1 = []
decorrelation_w1_md = []
msm_gen_jsd = []
msm_gen_jsd_md = []
msm_gen_w1 = []
msm_gen_w1_md = []

In [ ]:
no_tica = 0
no_angle = 0

for smile in smiles:
    if 'JSD_bond_angle_gen' not in data[smile].keys():
        no_angle += 1
        continue
    bond_angle_jsd += data[smile]['JSD_bond_angle_gen']
    bond_angle_jsd_md += data[smile]['JSD_bond_angle_md']
    bond_angle_w1 += data[smile]['W1_bond_angle_gen']
    bond_angle_w1_md += data[smile]['W1_bond_angle_md']
    bond_length_jsd += data[smile]['JSD_bond_length_gen']
    bond_length_jsd_md += data[smile]['JSD_bond_length_md']
    bond_length_w1 += data[smile]['W1_bond_length_gen']
    bond_length_w1_md += data[smile]['W1_bond_length_md']
    torsion_jsd += list(data[smile]['JSD_torsion_gen'].values())
    torsion_jsd_md += list(data[smile]['JSD_torsion_md'].values())
    torsion_w1 += list(data[smile]['W1_torsion_gen'].values())
    torsion_w1_md += list(data[smile]['W1_torsion_md'].values())
    
    decorrelation_gen = list(data[smile]['decorrelation_gen_in_ps'].values())
    decorrelation_ref = list(data[smile]['decorrelation_ref_in_ps'].values())
    decorrelation_md = list(data[smile]['decorrelation_md_in_ps'].values())

    if 'JSD_msm_gen' in data[smile].keys():
        msm_gen_jsd.append(data[smile]['JSD_msm_gen'])
        msm_gen_jsd_md.append(data[smile]['JSD_msm_md'])
        msm_gen_w1.append(data[smile]['W1_msm_gen'])
        msm_gen_w1_md.append(data[smile]['W1_msm_md'])
        
    for i in range(len(decorrelation_gen)):  # decorrelation for each torsion
        gen_p, gen_bin_edges = np.histogram(np.array(decorrelation_gen[i]), range=(5, 1380), bins=275)   # Because the stepsize if 5.2 ps
        ref_p, ref_bin_edges = np.histogram(np.array(decorrelation_ref[i]), range=(5, 1380), bins=275)
        md_p, md_bin_edges = np.histogram(np.array(decorrelation_md[i]), range=(5, 1380), bins=275)

        if np.sum(ref_p) == 0:
            continue
        gen_p = gen_p / np.sum(gen_p)
        ref_p = ref_p / np.sum(ref_p)
        md_p = md_p / np.sum(md_p)
        decorrelation_jsd.append(jensenshannon(ref_p, gen_p))
        decorrelation_jsd_md.append(jensenshannon(ref_p, md_p))
        decorrelation_w1.append(wasserstein_distance(decorrelation_gen[i], decorrelation_ref[i]))
        decorrelation_w1_md.append(wasserstein_distance(decorrelation_gen[i], decorrelation_md[i]))

    # sorted_jsd_torsion_gen_single_traj = []
    # for key, value in data[smile]['JSD_torsion_gen_single_traj'].items():
    #     sorted_jsd_torsion_gen_single_traj.append((key, np.mean(value)))
    # sorted_jsd_torsion_gen_single_traj.sort(key=lambda x: x[1])
    
    # for top_n in [1,3,5,10]:
    #     index = top_n - 1
    #     top_n_value = data[smile]['JSD_torsion_gen_single_traj'][sorted_jsd_torsion_gen_single_traj[index][0]]
    #     if top_n == 1:
    #         torsion_jsd_top1 += top_n_value
    #     elif top_n == 3:
    #         torsion_jsd_top3 += top_n_value
    #     elif top_n == 5:
    #         torsion_jsd_top5 += top_n_value
    #     elif top_n == 10:
    #         torsion_jsd_top10 += top_n_value

    # for top_n in [1,3,5,10]:
    #     index = top_n - 1
    #     top_n_value = data[smile]['JSD_bond_length_gen_single_traj'][sorted_jsd_torsion_gen_single_traj[index][0]]
    #     if top_n == 1:
    #         bond_length_jsd_top1 += top_n_value
    #     elif top_n == 3:
    #         bond_length_jsd_top3 += top_n_value
    #     elif top_n == 5:
    #         bond_length_jsd_top5 += top_n_value
    #     elif top_n == 10:
    #         bond_length_jsd_top10 += top_n_value

    # for top_n in [1,3,5,10]:
    #     index = top_n - 1
    #     top_n_value = data[smile]['JSD_bond_angle_gen_single_traj'][sorted_jsd_torsion_gen_single_traj[index][0]]
    #     if top_n == 1:
    #         bond_angle_jsd_top1 += [i for i in top_n_value if i != -1]  # -1 is the hard-coded value for bond angle error
    #     elif top_n == 3:
    #         bond_angle_jsd_top3 += [i for i in top_n_value if i != -1]
    #     elif top_n == 5:
    #         bond_angle_jsd_top5 += [i for i in top_n_value if i != -1]
    #     elif top_n == 10:
    #         bond_angle_jsd_top10 += [i for i in top_n_value if i != -1]

    if data[smile]['JSD_TICA_gen']:
        tica_0_jsd.append(data[smile]['JSD_TICA_gen']['TICA-0'])
        tica_0_jsd_md.append(data[smile]['JSD_TICA_md']['TICA-0'])
        tica_0_w1.append(data[smile]['W1_TICA_gen']['TICA-0'])
        tica_0_w1_md.append(data[smile]['W1_TICA_md']['TICA-0'])
        
        # for top_n in [1,3,5,10]:
        #     index = top_n - 1
        #     top_n_value = data[smile]['JSD_TICA_gen_single_traj']['TICA-0'][sorted_jsd_torsion_gen_single_traj[index][0]]
        #     if top_n == 1:
        #         tica_0_jsd_top1 += top_n_value
        #     elif top_n == 3:
        #         tica_0_jsd_top3 += top_n_value
        #     elif top_n == 5:
        #         tica_0_jsd_top5 += top_n_value
        #     elif top_n == 10:
        #         tica_0_jsd_top10 += top_n_value

        if 'TICA-0,1' in data[smile]['JSD_TICA_gen'].keys():
            tica_01_jsd.append(data[smile]['JSD_TICA_gen']['TICA-0,1'])
            tica_01_jsd_md.append(data[smile]['JSD_TICA_md']['TICA-0,1'])
            # for top_n in [1,3,5,10]:
            #     index = top_n - 1
            #     top_n_value = data[smile]['JSD_TICA_gen_single_traj']['TICA-0,1'][sorted_jsd_torsion_gen_single_traj[index][0]]
            #     if top_n == 1:
            #         tica_01_jsd_top1 += top_n_value
            #     elif top_n == 3:
            #         tica_01_jsd_top3 += top_n_value
            #     elif top_n == 5:
            #         tica_01_jsd_top5 += top_n_value
            #     elif top_n == 10:
            #         tica_01_jsd_top10 += top_n_value
    else:
        no_tica += 1
print('num of molecules with no tica:', no_tica)
print('num of molecules with no angle:', no_angle)

In [ ]:
# decorrelation_jsd has nan there, I want to compute mean without nan
decorrelation_jsd = np.array(decorrelation_jsd)
decorrelation_jsd = decorrelation_jsd[~np.isnan(decorrelation_jsd)]


In [ ]:
import csv

header = ['List Name', 'Mean (no md)', 'Std (no md)', 'Median (no md)', 'Min (no md)', 'Max (no md)', 
          'Mean (md)', 'Std (md)', 'Median (md)', 'Min (md)', 'Max (md)']
data_to_save = [
    ['bond_angle_jsd', round(np.mean(bond_angle_jsd), 5), round(np.std(bond_angle_jsd), 5), round(np.median(bond_angle_jsd), 5), round(np.min(bond_angle_jsd), 5), round(np.max(bond_angle_jsd), 5),
     round(np.mean(bond_angle_jsd_md), 5), round(np.std(bond_angle_jsd_md), 5), round(np.median(bond_angle_jsd_md), 5), round(np.min(bond_angle_jsd_md), 5), round(np.max(bond_angle_jsd_md), 5)],
    # ['bond_angle_jsd_top1', round(np.mean(bond_angle_jsd_top1), 5), round(np.std(bond_angle_jsd_top1), 5), round(np.median(bond_angle_jsd_top1), 5), round(np.min(bond_angle_jsd_top1), 5), round(np.max(bond_angle_jsd_top1), 5)],
    # ['bond_angle_jsd_top3', round(np.mean(bond_angle_jsd_top3), 5), round(np.std(bond_angle_jsd_top3), 5), round(np.median(bond_angle_jsd_top3), 5), round(np.min(bond_angle_jsd_top3), 5), round(np.max(bond_angle_jsd_top3), 5)],
    # ['bond_angle_jsd_top5', round(np.mean(bond_angle_jsd_top5), 5), round(np.std(bond_angle_jsd_top5), 5), round(np.median(bond_angle_jsd_top5), 5), round(np.min(bond_angle_jsd_top5), 5),round(np.max(bond_angle_jsd_top5))],
    # ['bond_angle_jsd_top10', round(np.mean(bond_angle_jsd_top10), 5), round(np.std(bond_angle_jsd_top10), 5), round(np.median(bond_angle_jsd_top10), 5), round(np.min(bond_angle_jsd_top10), 5), round(np.max(bond_angle_jsd_top10), 5)],
    ['bond_length_jsd', round(np.mean(bond_length_jsd), 5), round(np.std(bond_length_jsd), 5), round(np.median(bond_length_jsd), 5), round(np.min(bond_length_jsd), 5), round(np.max(bond_length_jsd), 5),
     round(np.mean(bond_length_jsd_md), 5), round(np.std(bond_length_jsd_md), 5), round(np.median(bond_length_jsd_md), 5), round(np.min(bond_length_jsd_md), 5), round(np.max(bond_length_jsd_md), 5)],
    # ['bond_length_jsd_top1', round(np.mean(bond_length_jsd_top1), 5), round(np.std(bond_length_jsd_top1), 5), round(np.median(bond_length_jsd_top1), 5), round(np.min(bond_length_jsd_top1), 5), round(np.max(bond_length_jsd_top1), 5)],
    # ['bond_length_jsd_top3', round(np.mean(bond_length_jsd_top3), 5), round(np.std(bond_length_jsd_top3), 5), round(np.median(bond_length_jsd_top3), 5), round(np.min(bond_length_jsd_top3), 5), round(np.max(bond_length_jsd_top3), 5)],    
    # ['bond_length_jsd_top5', round(np.mean(bond_length_jsd_top5), 5), round(np.std(bond_length_jsd_top5), 5), round(np.median(bond_length_jsd_top5), 5), round(np.min(bond_length_jsd_top5), 5), round(np.max(bond_length_jsd_top5), 5)],
    # ['bond_length_jsd_top10', round(np.mean(bond_length_jsd_top10), 5), round(np.std(bond_length_jsd_top10), 5), round(np.median(bond_length_jsd_top10), 5), round(np.min(bond_length_jsd_top10), 5), round(np.max(bond_length_jsd_top10), 5)],
    ['torsion_jsd', round(np.mean(torsion_jsd), 5), round(np.std(torsion_jsd), 5), round(np.median(torsion_jsd), 5), round(np.min(torsion_jsd), 5), round(np.max(torsion_jsd), 5),
     round(np.mean(torsion_jsd_md), 5), round(np.std(torsion_jsd_md), 5), round(np.median(torsion_jsd_md), 5), round(np.min(torsion_jsd_md), 5), round(np.max(torsion_jsd_md), 5)],
    # ['torsion_jsd_top1', round(np.mean(torsion_jsd_top1), 5), round(np.std(torsion_jsd_top1), 5), round(np.median(torsion_jsd_top1), 5), round(np.min(torsion_jsd_top1), 5), round(np.max(torsion_jsd_top1), 5)],
    # ['torsion_jsd_top3', round(np.mean(torsion_jsd_top3), 5), round(np.std(torsion_jsd_top3), 5), round(np.median(torsion_jsd_top3), 5), round(np.min(torsion_jsd_top3), 5), round(np.max(torsion_jsd_top3), 5)],
    # ['torsion_jsd_top5', round(np.mean(torsion_jsd_top5), 5), round(np.std(torsion_jsd_top5), 5), round(np.median(torsion_jsd_top5), 5), round(np.min(torsion_jsd_top5), 5), round(np.max(torsion_jsd_top5), 5)],
    # ['torsion_jsd_top10', round(np.mean(torsion_jsd_top10), 5), round(np.std(torsion_jsd_top10), 5), round(np.median(torsion_jsd_top10), 5), round(np.min(torsion_jsd_top10), 5), round(np.max(torsion_jsd_top10), 5)],
    ['tica_0_jsd', round(np.mean(tica_0_jsd), 5), round(np.std(tica_0_jsd), 5), round(np.median(tica_0_jsd), 5), round(np.min(tica_0_jsd), 5), round(np.max(tica_0_jsd), 5),
     round(np.mean(tica_0_jsd_md), 5), round(np.std(tica_0_jsd_md), 5), round(np.median(tica_0_jsd_md), 5), round(np.min(tica_0_jsd_md), 5), round(np.max(tica_0_jsd_md), 5)],
    # ['tica_0_jsd_top1', round(np.mean(tica_0_jsd_top1), 5), round(np.std(tica_0_jsd_top1), 5), round(np.median(tica_0_jsd_top1), 5), round(np.min(tica_0_jsd_top1), 5), round(np.max(tica_0_jsd_top1), 5)],
    # ['tica_0_jsd_top3', round(np.mean(tica_0_jsd_top3), 5), round(np.std(tica_0_jsd_top3), 5), round(np.median(tica_0_jsd_top3), 5), round(np.min(tica_0_jsd_top3), 5), round(np.max(tica_0_jsd_top3), 5)],
    # ['tica_0_jsd_top5', round(np.mean(tica_0_jsd_top5), 5), round(np.std(tica_0_jsd_top5), 5), round(np.median(tica_0_jsd_top5), 5), round(np.min(tica_0_jsd_top5), 5), round(np.max(tica_0_jsd_top5), 5)],
    # ['tica_0_jsd_top10', round(np.mean(tica_0_jsd_top10), 5), round(np.std(tica_0_jsd_top10), 5), round(np.median(tica_0_jsd_top10), 5), round(np.min(tica_0_jsd_top10), 5), round(np.max(tica_0_jsd_top10), 5)],
    ['tica_01_jsd', round(np.mean(tica_01_jsd), 5), round(np.std(tica_01_jsd), 5), round(np.median(tica_01_jsd), 5), round(np.min(tica_01_jsd), 5), round(np.max(tica_01_jsd), 5),
     round(np.mean(tica_01_jsd_md), 5), round(np.std(tica_01_jsd_md), 5), round(np.median(tica_01_jsd_md), 5), round(np.min(tica_01_jsd_md), 5), round(np.max(tica_01_jsd_md), 5)],
    # ['tica_01_jsd_top1', round(np.mean(tica_01_jsd_top1), 5), round(np.std(tica_01_jsd_top1), 5), round(np.median(tica_01_jsd_top1), 5), round(np.min(tica_01_jsd_top1), 5), round(np.max(tica_01_jsd_top1), 5)],
    # ['tica_01_jsd_top3', round(np.mean(tica_01_jsd_top3), 5), round(np.std(tica_01_jsd_top3), 5), round(np.median(tica_01_jsd_top3), 5), round(np.min(tica_01_jsd_top3), 5), round(np.max(tica_01_jsd_top3), 5)],
    # ['tica_01_jsd_top5', round(np.mean(tica_01_jsd_top5), 5), round(np.std(tica_01_jsd_top5), 5), round(np.median(tica_01_jsd_top5), 5), round(np.min(tica_01_jsd_top5), 5), round(np.max(tica_01_jsd_top5), 5)],
    # ['tica_01_jsd_top10', round(np.mean(tica_01_jsd_top10), 5), round(np.std(tica_01_jsd_top10), 5), round(np.median(tica_01_jsd_top10), 5), round(np.min(tica_01_jsd_top10), 5), round(np.max(tica_01_jsd_top10), 5)],
    ['torsion_w1', round(np.mean(torsion_w1), 5), round(np.std(torsion_w1), 5), round(np.median(torsion_w1), 5), round(np.min(torsion_w1), 5), round(np.max(torsion_w1), 5),
     round(np.mean(torsion_w1_md), 5), round(np.std(torsion_w1_md), 5), round(np.median(torsion_w1_md), 5), round(np.min(torsion_w1_md), 5), round(np.max(torsion_w1_md), 5)],
    ['bond_angle_w1', round(np.mean(bond_angle_w1), 5), round(np.std(bond_angle_w1), 5), round(np.median(bond_angle_w1), 5), round(np.min(bond_angle_w1), 5), round(np.max(bond_angle_w1), 5),
     round(np.mean(bond_angle_w1_md), 5), round(np.std(bond_angle_w1_md), 5), round(np.median(bond_angle_w1_md), 5), round(np.min(bond_angle_w1_md), 5), round(np.max(bond_angle_w1_md), 5)],
    ['bond_length_w1', round(np.mean(bond_length_w1), 5), round(np.std(bond_length_w1), 5), round(np.median(bond_length_w1), 5), round(np.min(bond_length_w1), 5), round(np.max(bond_length_w1), 5),
     round(np.mean(bond_length_w1_md), 5), round(np.std(bond_length_w1_md), 5), round(np.median(bond_length_w1_md), 5), round(np.min(bond_length_w1_md), 5), round(np.max(bond_length_w1_md), 5)],
    ['tica_0_w1', round(np.mean(tica_0_w1), 5), round(np.std(tica_0_w1), 5), round(np.median(tica_0_w1), 5), round(np.min(tica_0_w1), 5), round(np.max(tica_0_w1), 5),
     round(np.mean(tica_0_w1_md), 5), round(np.std(tica_0_w1_md), 5), round(np.median(tica_0_w1_md), 5), round(np.min(tica_0_w1_md), 5), round(np.max(tica_0_w1_md), 5)],
    ['decorrelation_jsd', round(np.mean(decorrelation_jsd), 5), round(np.std(decorrelation_jsd), 5), round(np.median(decorrelation_jsd), 5), round(np.min(decorrelation_jsd), 5), round(np.max(decorrelation_jsd), 5),
     round(np.mean(decorrelation_jsd_md), 5), round(np.std(decorrelation_jsd_md), 5), round(np.median(decorrelation_jsd_md), 5), round(np.min(decorrelation_jsd_md), 5), round(np.max(decorrelation_jsd_md), 5)],
    ['decorrelation_w1', round(np.mean(decorrelation_w1), 5), round(np.std(decorrelation_w1), 5), round(np.median(decorrelation_w1), 5), round(np.min(decorrelation_w1), 5), round(np.max(decorrelation_w1), 5),
        round(np.mean(decorrelation_w1_md), 5), round(np.std(decorrelation_w1_md), 5), round(np.median(decorrelation_w1_md), 5), round(np.min(decorrelation_w1_md), 5), round(np.max(decorrelation_w1_md), 5)],
    ['msm_gen_jsd', round(np.mean(msm_gen_jsd), 5), round(np.std(msm_gen_jsd), 5), round(np.median(msm_gen_jsd), 5), round(np.min(msm_gen_jsd), 5), round(np.max(msm_gen_jsd), 5),
        round(np.mean(msm_gen_jsd_md), 5), round(np.std(msm_gen_jsd_md), 5), round(np.median(msm_gen_jsd_md), 5), round(np.min(msm_gen_jsd_md), 5), round(np.max(msm_gen_jsd_md), 5)],
    ['msm_gen_w1', round(np.mean(msm_gen_w1), 5), round(np.std(msm_gen_w1), 5), round(np.median(msm_gen_w1), 5), round(np.min(msm_gen_w1), 5), round(np.max(msm_gen_w1), 5),
        round(np.mean(msm_gen_w1_md), 5), round(np.std(msm_gen_w1_md), 5), round(np.median(msm_gen_w1_md), 5), round(np.min(msm_gen_w1_md), 5), round(np.max(msm_gen_w1_md), 5)]

]

# Write to a CSV file
csv_file = f'{out_path}_statistics.csv'
with open(csv_file, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(header)
    writer.writerows(data_to_save)

print(f"Statistics saved to {csv_file}")

In [ ]:
# plot box-plot of bond_angle_jsd, bond_angle_jsd_md, bond_angle_w1, bond_angle_w1_md, torsion_jsd, torsion_jsd_md, torsion_w1, torsion_w1_md, tica_0_jsd, tica_0_jsd_md, tica_0_w1, tica_0_w1_md, tica_01_jsd, tica_01_jsd_md
import matplotlib.pyplot as plt
import pandas as pd
plt.figure(figsize=(18, 6))
plt.boxplot([bond_angle_jsd, bond_angle_jsd_md, bond_angle_w1, bond_angle_w1_md, torsion_jsd, torsion_jsd_md, torsion_w1, torsion_w1_md, tica_0_jsd, tica_0_jsd_md, tica_0_w1, tica_0_w1_md, tica_01_jsd, tica_01_jsd_md], labels=['bond_angle_jsd', 'bond_angle_jsd_md', 'bond_angle_w1', 'bond_angle_w1_md', 'torsion_jsd', 'torsion_jsd_md', 'torsion_w1', 'torsion_w1_md', 'tica_0_jsd', 'tica_0_jsd_md', 'tica_0_w1', 'tica_0_w1_md', 'tica_01_jsd', 'tica_01_jsd_md'])
plt.show()

In [ ]:
data['N[C@H](C(=O)O)[C@@H](N)CO']['W1_torsion_md']

In [ ]:
data['N[C@H](C(=O)O)[C@@H](N)CO']['JSD_torsion_gen']

In [ ]:
data['CCC(C#N)(CO)CO']['JSD_torsion_md']

In [ ]:
data['CCC(C#N)(CO)CO']['W1_bond_angle_gen']

In [ ]:
data['CCC(C#N)(CO)CO']['JSD_TICA_gen']

In [ ]:
np.array(list(data['C[C@@H](O)[C@@H](O)[C@@H](N)C#N']['decorrelation_ref_in_ps'].values()))

In [ ]:
data['CCC(C#N)(CO)CO']['JSD_bond_angle_gen_single_traj']